# FinQA Distill: SFT -> DPO -> Quick Benchmark

目标：基于已经产出的 `FinQA distill` 数据，单独完成一条精简训练链路：
- 不在 notebook 内重新跑 distill API 生成
- 仅处理 `FinQA`
- 先做 `SFT`
- 再做 `DPO`
- 每一阶段都用同一套 `benchmark` 做快速比较


## 0. Distill Outputs

这部分默认已经在命令行完成，不在本 notebook 内重复执行。

当前使用路径：
- `distill_sft.jsonl`: `/root/autodl-tmp/data/financial_reasoning/finqa_distill_r1_program_2048/distill_sft.jsonl`
- `distill_dpo.jsonl`: `/root/autodl-tmp/data/financial_reasoning/finqa_distill_r1_program_2048/distill_dpo.jsonl`


In [ ]:
from pathlib import Path
import json
import shlex
import subprocess
import pandas as pd

ROOT = Path('/root/MedicalGPT')
DATA_ROOT = Path('/root/autodl-tmp/data/financial_reasoning')
OUTPUT_ROOT = Path('/root/autodl-tmp/outputs/financial_reasoning')
BASE_MODEL = '/root/autodl-tmp/models/qwen/Qwen2___5-7B-Instruct'
TEMPLATE_NAME = 'qwen'

DISTILL_DIR = DATA_ROOT / 'finqa_distill_r1_program_2048'
DISTILL_SFT_RAW = DISTILL_DIR / 'distill_sft.jsonl'
DISTILL_DPO_RAW = DISTILL_DIR / 'distill_dpo.jsonl'
FINQA_EVAL_FILE = DATA_ROOT / 'raw' / 'finqa' / 'dev.json'

WORK_DIR = DATA_ROOT / 'clean' / 'distill_finqa_r1'
WORK_DIR.mkdir(parents=True, exist_ok=True)

SFT_NORMALIZED_FILE = WORK_DIR / 'train_distill_finqa_r1_normalized.jsonl'
SFT_CLEAN_FILE = WORK_DIR / 'train_distill_finqa_r1_clean.jsonl'
SFT_STRICT_FILE = WORK_DIR / 'train_distill_finqa_r1_strict.jsonl'
SFT_AUDIT_DIR = WORK_DIR / 'audit_sft'
SFT_TRAIN_DIR = WORK_DIR / 'sft_train_dir'
SFT_TRAIN_DIR.mkdir(parents=True, exist_ok=True)

DPO_CLEAN_FILE = WORK_DIR / 'train_distill_finqa_r1_dpo_clean.jsonl'
DPO_TRAIN_DIR = WORK_DIR / 'dpo_train_dir'
DPO_TRAIN_DIR.mkdir(parents=True, exist_ok=True)

SFT_OUT = OUTPUT_ROOT / 'distill_finqa_sft_lora'
SFT_MERGED_OUT = OUTPUT_ROOT / 'distill_finqa_sft_merged'
DPO_OUT = OUTPUT_ROOT / 'distill_finqa_dpo_lora'
BENCHMARK_SFT_DIR = OUTPUT_ROOT / 'benchmark_quick_distill_finqa_sft'
BENCHMARK_DPO_DIR = OUTPUT_ROOT / 'benchmark_quick_distill_finqa_dpo'

QUICK_FINQA_MAX_SAMPLES = 16
QUICK_MAX_NEW_TOKENS = 1024

for p in [DISTILL_DIR, WORK_DIR, SFT_TRAIN_DIR, DPO_TRAIN_DIR]:
    print(p, 'exists=' + str(p.exists()))


In [ ]:
def line_count(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())

preview = {
    'distill_sft_exists': DISTILL_SFT_RAW.exists(),
    'distill_sft_rows': line_count(DISTILL_SFT_RAW),
    'distill_dpo_exists': DISTILL_DPO_RAW.exists(),
    'distill_dpo_rows': line_count(DISTILL_DPO_RAW),
    'finqa_eval_exists': FINQA_EVAL_FILE.exists(),
}
print(json.dumps(preview, ensure_ascii=False, indent=2))


## 1. SFT

### 1.1 Distill SFT 数据清洗

这里使用 distill 专用流程：
1. `normalize_distill_sft.py`：把 assistant 侧从 `<think>/<answer>` 规范化成可训练版本
2. `clean_sharegpt_dataset.py`：做结构和长度过滤
3. `audit_distill_sft.py`：做 distill 专用脏样本审计
4. `filter_sharegpt_by_audit.py --mode strict`：产出最终 strict 训练集


In [ ]:
sft_clean_cmds = [
    [
        'python', '-m', 'data.normalize_distill_sft',
        '--input_file', str(DISTILL_SFT_RAW),
        '--output_file', str(SFT_NORMALIZED_FILE),
        '--assistant_mode', 'full',
    ],
    [
        'python', '-m', 'data.clean_sharegpt_dataset',
        '--source_file', str(SFT_NORMALIZED_FILE),
        '--output_file', str(SFT_CLEAN_FILE),
        '--min_turns', '2',
        '--max_turns', '2',
        '--max_total_chars', '6000',
        '--max_single_value_chars', '5000',
    ],
    [
        'python', '-m', 'data.audit_distill_sft',
        '--input_file', str(SFT_CLEAN_FILE),
        '--output_dir', str(SFT_AUDIT_DIR),
        '--max_answer_chars', '128',
    ],
    [
        'python', '-m', 'data.filter_sharegpt_by_audit',
        '--input_file', str(SFT_CLEAN_FILE),
        '--review_file', str(SFT_AUDIT_DIR / 'dirty_samples_review.jsonl'),
        '--output_file', str(SFT_STRICT_FILE),
        '--mode', 'strict',
    ],
]

for cmd in sft_clean_cmds:
    print(' '.join(shlex.quote(part) for part in cmd))

# for cmd in sft_clean_cmds:
#     result = subprocess.run(cmd, cwd=ROOT, check=True, capture_output=True, text=True)
#     print(result.stdout)

# (SFT_TRAIN_DIR / SFT_STRICT_FILE.name).write_text(SFT_STRICT_FILE.read_text(encoding='utf-8'), encoding='utf-8')


In [ ]:
if SFT_STRICT_FILE.exists():
    target = SFT_TRAIN_DIR / SFT_STRICT_FILE.name
    target.write_text(SFT_STRICT_FILE.read_text(encoding='utf-8'), encoding='utf-8')

sft_report = {
    'normalized_rows': line_count(SFT_NORMALIZED_FILE),
    'clean_rows': line_count(SFT_CLEAN_FILE),
    'strict_rows': line_count(SFT_STRICT_FILE),
    'train_dir_file': str(SFT_TRAIN_DIR / SFT_STRICT_FILE.name),
}
print(json.dumps(sft_report, ensure_ascii=False, indent=2))


### 1.2 SFT 训练

默认沿用 `run_fingpt_min.ipynb` 的 FinQA SFT 参数，单独输出到 `distill_finqa_sft_lora`。


In [ ]:
sft_cmd = [
    'python', '-m', 'training.supervised_finetuning',
    '--model_name_or_path', BASE_MODEL,
    '--tokenizer_name_or_path', BASE_MODEL,
    '--train_file_dir', str(SFT_TRAIN_DIR),
    '--validation_split_percentage', '1',
    '--do_eval',
    '--eval_steps', '100',
    '--eval_strategy', 'steps',
    '--do_train',
    '--use_peft',
    '--num_train_epochs', '2',
    '--per_device_train_batch_size', '1',
    '--max_grad_norm', '1.0',
    '--gradient_accumulation_steps', '16',
    '--gradient_checkpointing', 'True',
    '--warmup_ratio', '0.05',
    '--weight_decay', '0.05',
    '--learning_rate', '1e-5',
    '--logging_steps', '10',
    '--save_steps', '200',
    '--logging_first_step', 'True',
    '--report_to', 'tensorboard',
    '--logging_dir', str(OUTPUT_ROOT / 'tensorboard' / 'distill_finqa_sft'),
    '--model_max_length', '1024',
    '--target_modules', 'all',
    '--lora_rank', '8',
    '--lora_alpha', '16',
    '--lora_dropout', '0.05',
    '--torch_dtype', 'bfloat16',
    '--device_map', 'auto',
    '--output_dir', str(SFT_OUT),
    '--template_name', TEMPLATE_NAME,
]
print(' '.join(shlex.quote(part) for part in sft_cmd))
# subprocess.run(sft_cmd, cwd=ROOT, check=True)


In [ ]:
merge_sft_cmd = [
    'python', '-m', 'tooling.merge_peft_adapter',
    '--base_model', BASE_MODEL,
    '--tokenizer_path', BASE_MODEL,
    '--lora_model', str(SFT_OUT),
    '--output_dir', str(SFT_MERGED_OUT),
]
print(' '.join(shlex.quote(part) for part in merge_sft_cmd))
# subprocess.run(merge_sft_cmd, cwd=ROOT, check=True)


### 1.3 SFT benchmark 快速评估

这里只比较 `base vs sft`，并且只跑 `FinQA dev`。


In [ ]:
quick_benchmark_sft_cmd = [
    'python', '-m', 'evaluation.evaluate_financial_benchmarks',
    '--tokenizer_path', BASE_MODEL,
    '--model_entry', f'base={BASE_MODEL}',
    '--model_entry', f'sft={SFT_MERGED_OUT}',
    '--finqa_test_file', str(FINQA_EVAL_FILE),
    '--finqa_max_samples', str(QUICK_FINQA_MAX_SAMPLES),
    '--max_new_tokens', str(QUICK_MAX_NEW_TOKENS),
    '--temperature', '0.0',
    '--load_in_4bit',
    '--output_dir', str(BENCHMARK_SFT_DIR),
]
print(' '.join(shlex.quote(part) for part in quick_benchmark_sft_cmd))
# subprocess.run(quick_benchmark_sft_cmd, cwd=ROOT, check=True)


In [ ]:
summary_csv = BENCHMARK_SFT_DIR / 'benchmark_summary.csv'
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    display(df[df['task_name'].isin(['finqa_test', 'bucket_finqa', 'macro_average'])])
else:
    print('Missing:', summary_csv)


## 2. DPO

### 2.1 Distill DPO 数据清洗

这里不重新构造 pair，只对已经产出的 `distill_dpo.jsonl` 做规范化和结构过滤。


In [ ]:
dpo_clean_cmd = [
    'python', '-m', 'data.normalize_distill_dpo',
    '--input_file', str(DISTILL_DPO_RAW),
    '--output_file', str(DPO_CLEAN_FILE),
]
print(' '.join(shlex.quote(part) for part in dpo_clean_cmd))
# subprocess.run(dpo_clean_cmd, cwd=ROOT, check=True)


In [ ]:
if DPO_CLEAN_FILE.exists():
    target = DPO_TRAIN_DIR / DPO_CLEAN_FILE.name
    target.write_text(DPO_CLEAN_FILE.read_text(encoding='utf-8'), encoding='utf-8')

print(json.dumps({
    'dpo_clean_rows': line_count(DPO_CLEAN_FILE),
    'dpo_train_dir_file': str(DPO_TRAIN_DIR / DPO_CLEAN_FILE.name),
}, ensure_ascii=False, indent=2))


### 2.2 DPO 训练

默认使用 `distill_finqa_sft_merged` 作为 base，再挂载 DPO LoRA。


In [ ]:
dpo_cmd = [
    'python', '-m', 'training.dpo_training',
    '--model_name_or_path', str(SFT_MERGED_OUT),
    '--tokenizer_name_or_path', BASE_MODEL,
    '--template_name', TEMPLATE_NAME,
    '--train_file_dir', str(DPO_TRAIN_DIR),
    '--validation_split_percentage', '1',
    '--do_train',
    '--use_peft', 'True',
    '--per_device_train_batch_size', '1',
    '--gradient_accumulation_steps', '16',
    '--gradient_checkpointing', 'True',
    '--learning_rate', '5e-7',
    '--max_steps', '200',
    '--max_source_length', '1024',
    '--max_target_length', '512',
    '--logging_steps', '10',
    '--save_steps', '200',
    '--target_modules', 'all',
    '--lora_rank', '8',
    '--lora_alpha', '16',
    '--lora_dropout', '0.05',
    '--torch_dtype', 'float16',
    '--device_map', 'auto',
    '--output_dir', str(DPO_OUT),
]
print(' '.join(shlex.quote(part) for part in dpo_cmd))
# subprocess.run(dpo_cmd, cwd=ROOT, check=True)


### 2.3 DPO benchmark 快速评估

这里比较 `base vs sft vs dpo`，仍然只跑 `FinQA dev`。


In [ ]:
quick_benchmark_dpo_cmd = [
    'python', '-m', 'evaluation.evaluate_financial_benchmarks',
    '--tokenizer_path', BASE_MODEL,
    '--model_entry', f'base={BASE_MODEL}',
    '--model_entry', f'sft={SFT_MERGED_OUT}',
    '--model_entry', f'dpo={SFT_MERGED_OUT}',
    '--adapter_entry', f'dpo={DPO_OUT}',
    '--finqa_test_file', str(FINQA_EVAL_FILE),
    '--finqa_max_samples', str(QUICK_FINQA_MAX_SAMPLES),
    '--max_new_tokens', str(QUICK_MAX_NEW_TOKENS),
    '--temperature', '0.0',
    '--load_in_4bit',
    '--output_dir', str(BENCHMARK_DPO_DIR),
]
print(' '.join(shlex.quote(part) for part in quick_benchmark_dpo_cmd))
# subprocess.run(quick_benchmark_dpo_cmd, cwd=ROOT, check=True)


In [ ]:
summary_csv = BENCHMARK_DPO_DIR / 'benchmark_summary.csv'
if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    display(df[df['task_name'].isin(['finqa_test', 'bucket_finqa', 'macro_average'])])
else:
    print('Missing:', summary_csv)


## Notes

- 这个 notebook 默认只生成命令，不自动开始训练；把对应单元里的 `subprocess.run(...)` 取消注释即可执行。
- `SFT` 阶段为了避免误杀短答案，审计使用的是 `audit_distill_sft.py`，不是原来的通用审计脚本。
- `DPO` 阶段保留 `<think>/<answer>`，因为偏好学习本来就需要比较 reasoning 质量。
- 当前 quick benchmark 只跑 `FinQA`，方便先看 distill 主链路是否有效。
